# Objective
In this notebook, I am developing the logic for calculating state durations, focusing on SIRS as an example, and filter out short or long duration based on the required logic.<br>

Choosing the threshold is a manual process based on the data histogram.<br>

There are two thresholds to be selected: 
1. Threshold for the negative state in order to merge short negative state with the prev and next positive states
2. Thresold for the positive state, which should be selected after applying the merge logic, to remove the short states and decrease the number of false positives.


In [18]:
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns
import numpy as np
import datetime as dt
import os

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src_strategy.configs.aggregator import FEATURE_REGISTRY
from src_strategy.configs.sirscalculator import sirs_config
from src_strategy.configs.suspected_infection import suspected_infection_config
from src_strategy.events.sirs import build_sirs_pipeline
from src_strategy.events.suspected_infection import build_suspected_infection_pipeline

import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style="whitegrid")

In [16]:
os.listdir(input_output_config_3_1.data_path)

['Lab Results - 6.5.26.csv',
 'Procedure Order Events - 6.5.26.csv',
 'Med Admin Events - 6.8.26.csv',
 'flowsheets_bp - 6.8.26.csv',
 'labs_with_fio2 - 6.9.26.csv',
 'Med Admin Events - 6.5.26.csv',
 'Blood Pressure - 6.8.26.csv',
 'Export Log.xlsx',
 'Encounter Table with Baseline Values - June 2022 - May 2026 - 6.5.26.csv',
 'Diagnoses - 6.5.26.csv',
 'Flowsheet Events - 6.5.26.csv',
 'FIO2 - 6.9.26.csv']

In [38]:
from src_strategy.configs.dataconfig import input_output_config_3_1
OUTPUT_DIR = Path(input_output_config_3_1.output_path)
INPUT_DIR = Path(input_output_config_3_1.data_path)
SIRS_PATH = OUTPUT_DIR / "df_sirs.parquet"
ENCOUNTER_PATH = INPUT_DIR / "Encounter Table with Baseline Values - June 2022 - May 2026 - 6.5.26.csv"
AGGREGATED_PATH = OUTPUT_DIR / "df_aggregated.parquet"
INFECTION_PATH = OUTPUT_DIR / "df_suspected_infection.parquet"
EVENT_PATH = OUTPUT_DIR / "df_all_no_collisions.parquet"


# Data 
1. df_sirs.parquet: This file contains the SIRS state information for each patient over time. The SIRS state is represented as a binary variable, where 1 indicates the presence of SIRS and 0 indicates its absence. The data is structured in a time-series format, with each row representing a specific time point for a patient.
2. df_suspected_infection.parquet: This file contains information about suspected infections for each patient. It includes details such as the type of infection, the time of onset, and other relevant clinical data. This data is also structured in a time-series format, allowing for the analysis of infection patterns over time.


In [39]:
df_encs = pl.read_csv(ENCOUNTER_PATH)

In [34]:
df_all = pl.read_parquet(EVENT_PATH)
df_sirs = pl.read_parquet(SIRS_PATH)
df_infection = pl.read_parquet(INFECTION_PATH)

# Missing encounters from encounters table (Irrelevant to the notebook)

In [43]:
df_encs['EncounterEpicCsn'].n_unique(), df_all['EncounterEpicCsn'].n_unique()

(14914, 14932)

In [45]:
encs = set(df_all['EncounterEpicCsn'].unique())-set(df_encs['EncounterEpicCsn'].unique())

In [50]:
df_all.filter(pl.col("EncounterEpicCsn").is_in(encs)).group_by("EncounterEpicCsn").agg(
	pl.col("Event_DateTime").min().alias("first_event_dt"),
	pl.col("Event_DateTime").max().alias("last_event_dt"),
	pl.len().alias('n_rows')
).with_columns(
	(pl.col("last_event_dt")-pl.col("first_event_dt")).dt.total_hours(fractional=True).alias("duration")
).describe()

statistic,EncounterEpicCsn,first_event_dt,last_event_dt,n_rows,duration
str,f64,str,str,f64,f64
"""count""",18.0,"""18""","""18""",18.0,18.0
"""null_count""",0.0,"""0""","""0""",0.0,0.0
"""mean""",7.5284e8,"""2026-05-20 05:14:00.166666""","""2026-05-28 02:08:36.888889""",85.388889,188.910201
"""std""",1.5724e6,null,null,89.907887,150.233183
"""min""",7.4701396e8,"""2026-04-28 18:16:00""","""2026-04-28 18:16:00""",1.0,0.0
"""25%""",7.52969002e8,"""2026-05-16 11:54:00""","""2026-05-27 08:52:34""",28.0,91.167778
"""50%""",7.53159771e8,"""2026-05-24 21:19:00""","""2026-05-30 11:10:04""",56.0,113.912778
"""75%""",7.53663649e8,"""2026-05-26 17:49:00""","""2026-05-31 15:16:00""",91.0,299.566667
"""max""",7.53774459e8,"""2026-06-02 07:45:00""","""2026-06-06 15:24:00""",332.0,498.325556


# Logic
1. Calculate the duration of each state (positive and negative) for each patient.
2. Filter out the episodes that are not associated with infection (i.e., those that do not have a corresponding entry in the suspected infection table).
3. Plot the histogram of the durations of the negative episodes to determine a threshold for merging short negative states with the surrounding positive states.
4. Apply the merging logic to combine short negative states with the previous and next positive states.
5. After merging, plot the histogram of the durations of the positive episodes to determine a threshold
6. Apply the threshold to filter out short positive states, reducing the number of false positives.

### 1. Calculate the duration of each state (positive and negative) for each patient.

In [29]:
all_episodes = (
	df_sirs.select("EncounterEpicCsn", "Event_DateTime", "sirs_score")
	.sort(by=["EncounterEpicCsn", "Event_DateTime"])
	.with_columns(
		(pl.col("sirs_score")>=2).alias("sirs_positive")
	)
	.with_columns(
		pl.col("sirs_positive").ne(
			pl.col("sirs_positive").shift(1).over("EncounterEpicCsn")
		).fill_null(True).alias('_flip')
	)
	.with_columns(
		pl.col("_flip").cast(pl.Int32).cum_sum().over("EncounterEpicCsn").alias("episode_id")
	)
	.group_by(["EncounterEpicCsn", "episode_id"], maintain_order=True)
	.agg(
		pl.col("sirs_positive").first(),
		pl.col("Event_DateTime").first().alias("episode_begin_time"),
		pl.col("Event_DateTime").last().alias("episode_last_time"),
		pl.col('sirs_score').min().alias("min_sirs_score"),
		pl.col('sirs_score').max().alias("max_sirs_score"),
		pl.len().alias("n_rows")
	)
	.with_columns(
		pl.coalesce([
			pl.col("episode_begin_time").shift(-1).over("EncounterEpicCsn"),
			pl.col("episode_last_time")
		]).alias("episode_end_time")
	)
	.with_columns(
		(pl.col("episode_end_time")-pl.col("episode_begin_time")).dt.total_minutes(fractional=True).alias('episode_duration_min')
	)
)

all_episodes = all_episodes.drop([c for c in all_episodes.columns if c.startswith('_')])

In [30]:
all_episodes

EncounterEpicCsn,episode_id,sirs_positive,episode_begin_time,episode_last_time,min_sirs_score,max_sirs_score,n_rows,episode_end_time,episode_duration_min
i64,i32,bool,datetime[μs],datetime[μs],i64,i64,u64,datetime[μs],f64
659308243,1,false,2022-06-14 10:50:00,2022-06-14 22:05:00,0,1,17,2022-06-14 22:10:00,680.0
659308243,2,true,2022-06-14 22:10:00,2022-06-14 22:45:00,2,2,9,2022-06-14 23:00:00,50.0
659308243,3,false,2022-06-14 23:00:00,2022-06-14 23:00:00,1,1,1,2022-06-14 23:15:00,15.0
659308243,4,true,2022-06-14 23:15:00,2022-06-14 23:30:00,2,2,2,2022-06-14 23:45:00,30.0
659308243,5,false,2022-06-14 23:45:00,2022-06-14 23:45:00,1,1,1,2022-06-15 00:00:00,15.0
…,…,…,…,…,…,…,…,…,…
753665697,5,false,2026-05-27 15:42:35,2026-06-05 02:06:26.310,0,1,19,2026-06-05 02:06:26.310,12143.855167
753676594,1,false,2026-05-26 17:49:00,2026-05-31 08:55:00,0,0,91,2026-05-31 08:55:00,6666.0
753679492,1,false,2026-05-26 20:20:00,2026-05-30 07:32:00,0,0,27,2026-05-30 07:32:00,4992.0


In [36]:
all_episodes.filter(
	pl.col('episode_duration_min')<=0
).sort("episode_id")

EncounterEpicCsn,episode_id,sirs_positive,episode_begin_time,episode_last_time,min_sirs_score,max_sirs_score,n_rows,episode_end_time,episode_duration_min
i64,i32,bool,datetime[μs],datetime[μs],i64,i64,u64,datetime[μs],f64
751544176,1,false,2026-04-28 18:16:00,2026-04-28 18:16:00,0,0,1,2026-04-28 18:16:00,0.0
682037212,5,false,2023-05-25 02:07:53.940,2023-05-25 02:07:53.940,0,0,1,2023-05-25 02:07:53.940,0.0
664288872,11,false,2022-07-07 02:08:11.227,2022-07-07 02:08:11.227,0,0,1,2022-07-07 02:08:11.227,0.0
711980653,15,false,2024-10-03 03:56:59.727,2024-10-03 03:56:59.727,0,0,1,2024-10-03 03:56:59.727,0.0
692219780,19,false,2023-11-17 02:53:04.007,2023-11-17 02:53:04.007,0,0,1,2023-11-17 02:53:04.007,0.0
…,…,…,…,…,…,…,…,…,…
666718670,215,false,2022-09-17 02:38:41.143,2022-09-17 02:38:41.143,0,0,1,2022-09-17 02:38:41.143,0.0
674025842,251,false,2023-01-18 02:17:09.440,2023-01-18 02:17:09.440,0,0,1,2023-01-18 02:17:09.440,0.0
679683856,391,false,2023-05-26 01:59:09.327,2023-05-26 01:59:09.327,0,0,1,2023-05-26 01:59:09.327,0.0


In [51]:
all_episodes.describe()

statistic,EncounterEpicCsn,episode_id,sirs_positive,episode_begin_time,episode_last_time,min_sirs_score,max_sirs_score,n_rows,episode_end_time,episode_duration_min
str,f64,f64,f64,str,str,f64,f64,f64,str,f64
"""count""",654363.0,654363.0,654363.0,"""654363""","""654363""",654363.0,654363.0,654363.0,"""654363""",654363.0
"""null_count""",0.0,0.0,0.0,"""0""","""0""",0.0,0.0,0.0,"""0""",0.0
"""mean""",7.0672e8,147.168374,0.489945,"""2024-07-05 01:35:57.195713""","""2024-07-05 23:21:44.856671""",1.34448,1.579127,12.271385,"""2024-07-06 00:01:32.262050""",1345.584439
"""std""",2.5096e7,315.572011,null,null,null,0.736802,0.703426,38.373143,null,14779.165995
"""min""",6.59308243e8,1.0,0.0,"""2022-05-05 02:05:03.487000""","""2022-06-01 05:35:00""",0.0,0.0,1.0,"""2022-06-01 06:00:00""",0.0
"""25%""",6.84967527e8,16.0,null,"""2023-08-03 07:39:00""","""2023-08-04 06:30:00""",1.0,1.0,1.0,"""2023-08-04 07:15:00""",19.0
"""50%""",7.05197488e8,53.0,null,"""2024-07-13 10:53:00""","""2024-07-14 10:00:00""",1.0,1.0,3.0,"""2024-07-14 11:00:00""",60.0
"""75%""",7.27928303e8,149.0,null,"""2025-06-16 15:13:00""","""2025-06-17 11:12:00""",2.0,2.0,9.0,"""2025-06-17 11:43:00""",282.0
"""max""",7.53774459e8,4449.0,1.0,"""2026-06-03 02:19:37.180000""","""2026-06-06 15:24:00""",4.0,4.0,5489.0,"""2026-06-06 15:24:00""",1.6449e6


### 2. Filter out the episodes that are not associated with infection (i.e., those that do not have a corresponding entry in the suspected infection table).

### TODO
#TODO
1. Infection association checks only the episode start. It should check whether the episode interval overlaps the infection window.
2. Gap detection then runs on this duplicated joined table, partitioned only by encounter. Therefore shift() may not refer to the actual preceding/following SIRS episode

In [ ]:
HOURS = 24
episodes_around_infection = df_infection.join(
	all_episodes,
	on="EncounterEpicCsn",
	how='left'
).filter(
	(pl.col("episode_begin_time")-pl.col("infect_dt")).abs().dt.total_hours(fractional=True) < HOURS
)

### 3. Plot the histogram of the durations of the negative episodes to determine a threshold for merging short negative states with the surrounding positive states.

In [71]:
SHORT_DEF = 6 # hours
BIN_WIDTH = 5 # minutes

neg_short_episodes = episodes_around_infection.filter(
	~pl.col("sirs_positive")
	&(pl.col("episode_duration_min") < SHORT_DEF*60)
)

max_val = int(neg_short_episodes['episode_duration_min'].max())
print(max_val)
if max_val%BIN_WIDTH != 0:
	max_val = max_val + (BIN_WIDTH - max_val%BIN_WIDTH)

counts, bins = np.histogram(
	neg_short_episodes['episode_duration_min'].to_pandas(),
	bins=np.arange(0, max_val+BIN_WIDTH, BIN_WIDTH)
)

bins_center = (bins[:-1]+bins[1:]) / 2
px.bar(x=bins_center, y=counts)

359


### 4. Apply the merging logic to combine short negative states with the previous and next positive states.

### Decision
Threshold for the bin at x=17.5. `x<=20`

In [78]:
neg_short_episodes
NEG_DUR_THRESHOLD = 20 # minutes

In [87]:
mark = (
	episodes_around_infection.with_columns(
		pl.col("sirs_positive").shift(1).over("EncounterEpicCsn").fill_null(False).alias("_prev_sirs_positive"),
		pl.col("sirs_positive").shift(-1).over("EncounterEpicCsn").fill_null(False).alias("_next_sirs_positive")
	)
	.with_columns(
		(pl.col("_prev_sirs_positive") & ~pl.col("sirs_positive") & pl.col("_next_sirs_positive")  
		&(pl.col("episode_duration_min") < NEG_DUR_THRESHOLD )).alias("_gap")
	)
	.with_columns(
		(pl.col("_gap") | (pl.col("_gap").shift().over("EncounterEpicCsn"))).alias("_continuous_gap")
	)
	.with_columns(
		((~pl.col("_continuous_gap")).fill_null(True).cum_sum().over("EncounterEpicCsn")).alias("merge_id")
	)
)
mark

EncounterEpicCsn,criterion,infect_dt,suspicion_infection_type,episode_id,sirs_positive,episode_begin_time,episode_last_time,min_sirs_score,max_sirs_score,n_rows,episode_end_time,episode_duration_min,_prev_sirs_positive,_next_sirs_positive,_gap,_continuous_gap,merge_id
i64,str,datetime[μs],str,i32,bool,datetime[μs],datetime[μs],i64,i64,u64,datetime[μs],f64,bool,bool,bool,bool,u32
659308243,"""culture_dt""",2022-06-15 02:03:00,"""IV+Culture""",1,false,2022-06-14 10:50:00,2022-06-14 22:05:00,0,1,17,2022-06-14 22:10:00,680.0,false,true,false,null,1
659308243,"""culture_dt""",2022-06-15 02:03:00,"""IV+Culture""",2,true,2022-06-14 22:10:00,2022-06-14 22:45:00,2,2,9,2022-06-14 23:00:00,50.0,false,false,false,false,2
659308243,"""culture_dt""",2022-06-15 02:03:00,"""IV+Culture""",3,false,2022-06-14 23:00:00,2022-06-14 23:00:00,1,1,1,2022-06-14 23:15:00,15.0,true,true,true,true,2
659308243,"""culture_dt""",2022-06-15 02:03:00,"""IV+Culture""",4,true,2022-06-14 23:15:00,2022-06-14 23:30:00,2,2,2,2022-06-14 23:45:00,30.0,false,false,false,true,2
659308243,"""culture_dt""",2022-06-15 02:03:00,"""IV+Culture""",5,false,2022-06-14 23:45:00,2022-06-14 23:45:00,1,1,1,2022-06-15 00:00:00,15.0,true,true,true,true,2
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
753665697,"""iv_dt""",2026-05-26 18:25:36,"""IV+Culture""",1,false,2026-05-26 15:40:17,2026-05-26 17:24:00,1,1,3,2026-05-26 17:42:00,121.716667,false,true,false,false,6
753665697,"""iv_dt""",2026-05-26 18:25:36,"""IV+Culture""",2,true,2026-05-26 17:42:00,2026-05-27 02:15:00,2,2,10,2026-05-27 02:30:00,528.0,false,false,false,false,7
753665697,"""iv_dt""",2026-05-26 18:25:36,"""IV+Culture""",3,false,2026-05-27 02:30:00,2026-05-27 10:48:27,0,1,13,2026-05-27 13:00:00,630.0,true,true,false,false,8


In [94]:
mark['infect_dt'].n_unique()

37491

In [101]:
    # Collapse every connected positive-gap-positive chain into one row.
merged_positive = (
        mark
        .group_by("EncounterEpicCsn", "merge_id", maintain_order=True)
        .agg(
            # True when the group contains at least one positive episode.
            pl.col("sirs_positive").any(),

            pl.col("infect_dt").min(),
            pl.col("episode_begin_time").min(),
            pl.col("episode_end_time").max(),
            pl.col("min_sirs_score").min(),
            pl.col("max_sirs_score").max(),
            pl.col("n_rows").sum(),

            # Number of short negative gaps absorbed into the episode.
            pl.col("_gap")
            .sum()
            .alias("bridged_gap_count"),
        )

        # Recalculate duration across the complete merged interval.
        .with_columns(
            (
                pl.col("episode_end_time")
                - pl.col("episode_begin_time")
            )
            .dt.total_minutes(fractional=True)
            .alias("episode_duration_min")
        )

        # Remove standalone negative episodes.
        .filter("sirs_positive")
        .rename({"merge_id": "episode_id"})
    )

merged_positive

EncounterEpicCsn,episode_id,sirs_positive,infect_dt,episode_begin_time,episode_end_time,min_sirs_score,max_sirs_score,n_rows,bridged_gap_count,episode_duration_min
i64,u32,bool,datetime[μs],datetime[μs],datetime[μs],i64,i64,u64,u64,f64
659308243,2,true,2022-06-15 02:03:00,2022-06-14 22:10:00,2022-06-15 01:15:00,1,2,20,3,185.0
659308243,4,true,2022-06-15 02:03:00,2022-06-15 05:00:00,2022-06-15 06:30:00,2,2,7,0,90.0
659308243,6,true,2022-06-15 02:03:00,2022-06-15 07:30:00,2022-06-15 07:45:00,2,2,1,0,15.0
659308243,8,true,2022-06-15 02:03:00,2022-06-15 10:44:00,2022-06-15 22:35:00,1,2,51,3,711.0
659308243,10,true,2022-06-15 02:03:00,2022-06-16 00:00:00,2022-06-16 03:00:00,2,2,4,0,180.0
…,…,…,…,…,…,…,…,…,…,…
753579730,16,true,2026-05-26 00:35:03,2026-05-26 17:00:00,2026-05-27 00:00:00,2,2,6,0,420.0
753665697,2,true,2026-05-26 17:24:00,2026-05-26 17:42:00,2026-05-27 02:30:00,2,2,10,0,528.0
753665697,4,true,2026-05-26 17:24:00,2026-05-27 13:00:00,2026-05-27 15:42:35,2,2,2,0,162.583333


In [95]:
merged_positive['infect_dt'].n_unique()

34540

### 5. After merging, plot the histogram of the durations of the positive episodes to determine a threshold

In [102]:
merged_positive.filter(pl.col('episode_duration_min') == 0)

EncounterEpicCsn,episode_id,sirs_positive,infect_dt,episode_begin_time,episode_end_time,min_sirs_score,max_sirs_score,n_rows,bridged_gap_count,episode_duration_min
i64,u32,bool,datetime[μs],datetime[μs],datetime[μs],i64,i64,u64,u64,f64


In [104]:
BINS_WIDTH = 5
max_duration = int(merged_positive.filter(pl.col('episode_duration_min') < 12*24)['episode_duration_min'].max())

if max_duration%BINS_WIDTH != 0:
	max_duration = max_duration + (BINS_WIDTH - max_duration%BINS_WIDTH)

counts, bins = np.histogram(
	merged_positive.filter(pl.col('episode_duration_min') < 12*24)['episode_duration_min'].to_pandas(),
	bins=np.arange(0, max_duration+BINS_WIDTH, BINS_WIDTH)
)

bins_center = (bins[:-1]+bins[1:]) / 2
px.bar(x=bins_center, y=counts)


### Decision 
Threshold for the bin at x=17.5. `x<=20`

In [105]:
merged_positive_filtered = merged_positive.filter(
	pl.col("episode_duration_min") > 20
)

In [127]:
sepsis_1_filtered = merged_positive_filtered.with_columns(
	pl.min_horizontal([pl.col("episode_begin_time"), pl.col("infect_dt")]).alias("sepsis_1_dt"),
).group_by("EncounterEpicCsn").agg(pl.col("sepsis_1_dt").min().alias("sepsis_1_dt")).join(
	df_encs.select("EncounterEpicCsn", pl.col("Arrival_Instant").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f")),
	on="EncounterEpicCsn",
	how="left"
).with_columns(
	((pl.col("sepsis_1_dt")-pl.col("Arrival_Instant")).dt.total_hours(fractional=True) <= 48).alias("POA")
)

/tmp/ipykernel_726851/3818588598.py:4: ChronoFormatWarning:

Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime



In [109]:
df_encs

PrimaryMrn,PatientAgeAtAdmission,Sex,Ethnicity,FirstRace,MultiRacial,EncounterEpicCsn,AdmissionDateValue,DischargeDateValue,Arrival_Instant,FirstAdmissionOrderInstant,InpatientAdmissionInstant,Admitted_from_ED,PatientClass,InpatientAdmissionPatientClass,HospitalService,LengthOfStayInDays,AdmittingDepartment,DischargeDepartment,AdmissionType,AdmissionSource,AdmissionOrigin,PrincipalProblem,PrimaryCodedDiagnosis,PrimaryCodedProcedureKey,Death_Flag,Original_POA_Condition,Sepsis_Category,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_Creatinine,Baseline_Platelets,Baseline_Bilirubin,Baseline_eGFR,Baseline_WBC,First_Suspected_Infection_Time,Suspected_Infection_Criteria_Met,Cancer_Registry_YN,HIV_Registry_YN,Immunocrompromised_Registry_YN,CKD_Dialysis_Registry_YN,Solid_Organ_Transplant_Registry_YN,Pregnancy_Registry_YN
i64,f64,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64
93085941,61.954825,"""Male""","""Non-Hispanic/Latino""","""White""",0,687038913,"""2023-08-15""","""2023-09-06""","""2023-08-15 08:19:00.0000000""","""2023-08-15 13:35:00.000""","""2023-08-15 13:35:00.0000000""","""No""","""Inpatient""","""Outpatient""","""Hospital Medicine""",22,"""ZL 3RD FL (ICU)""","""UH 08G""","""Urgent""","""Short Term General Hospital""","""Transfer Center Admission""","""Acute renal failure superimpos…","""Sepsis, unspecified organism""","""Insertion of Monitoring Device…",0,"""POA-3""","""POA-3""","""161""","""16""","""57""","""3.96280701754386""","""153.625""","""1.99047619047619""","""21.9615384615385""","""12.0849230769231""","""2023-08-15 09:03:00.0000000""","""Code Sepsis Order""",0,0,0,0,0,0
94630920,69.284052,"""Male""","""Unknown""","""Unavailable/Unknown""",0,697150447,"""2024-01-26""","""2024-01-31""","""2024-01-26 13:38:00.0000000""","""2024-01-26 12:13:00.000""","""2024-01-26 13:38:00.0000000""","""No""","""Inpatient""","""Inpatient""","""Physical Medicine & Rehabilita…",5,"""ZL 8TH FL""","""ZL 8TH FL""","""Elective""","""Short Term General Hospital""","""Transfer Center Admission""","""Status post total replacement …","""Aftercare following joint repl…","""*Unspecified""",0,"""NPOA-1""","""NPOA-1""","""123""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""2024-01-30 23:32:00.0000000""","""Lactate + Two Blood Culture Or…",0,0,0,0,0,0
90121567,74.787132,"""Male""","""Non-Hispanic/Latino""","""White""",0,662909282,"""2022-06-01""","""2022-06-11""","""2022-06-01 01:15:00.0000000""","""2022-06-01 03:59:00.000""","""2022-06-01 03:59:00.0000000""","""Yes""","""Inpatient""","""Inpatient""","""Hospital Medicine""",10,"""UH 07B ICU""","""UH 10G""","""Emergency""","""Home & Outside Location""","""ED Admission""","""Sepsis, due to unspecified org…","""Sepsis, unspecified organism""","""Insertion of Monitoring Device…",0,"""POA-3""","""POA-3""","""115""","""16""","""61""","""0.617777777777778""","""291.703703703704""","""0.7""","""99""","""12.17""","""2022-06-01 01:20:00.0000000""","""Suspected Infection Flowsheet""",1,0,0,0,0,0
92133040,68.958247,"""Female""","""Non-Hispanic/Latino""","""Southeast Asian (Thailand, Vie…",0,683097675,"""2023-06-02""","""2023-06-04""","""2023-06-02 02:10:00.0000000""","""2023-06-02 03:56:00.000""","""2023-06-02 02:10:00.0000000""","""No""","""Inpatient""","""Inpatient""","""Hospital Medicine-ONC""",2,"""UH 11G ONC""","""UH 11G ONC""","""Emergency""","""Short Term General Hospital""","""Transfer Center Admission""","""Community acquired pneumonia o…","""Sepsis, unspecified organism""","""*Unspecified""",0,"""POA-2""","""POA-2""","""99""","""16""","""85""","""0.7""","""143.666666666667""","""0.483333333333333""","""90.75""","""4.26666666666667""","""NULL""","""NULL""",1,0,0,0,0,0
97140404,79.479808,"""Male""","""Non-Hispanic/Latino""","""White""",0,683211004,"""2023-06-05""","""2023-06-08""","""2023-06-05 10:38:00.0000000""","""2023-06-05 15

In [137]:
sepsis_1_raw = df_sirs.filter(pl.col("sirs_score")>=2).join(
	df_infection,
	on="EncounterEpicCsn",
	how="left"
).filter(
	(pl.col("Event_DateTime")-pl.col("infect_dt")).abs().dt.total_hours(fractional=True) <= 24 
).with_columns(
	pl.min_horizontal([pl.col("Event_DateTime"), pl.col("infect_dt")]).alias("sepsis_1_dt")
).group_by(
	"EncounterEpicCsn"
).agg(
	pl.col("sepsis_1_dt").min().alias("sepsis_1_dt")
).join(
	df_encs.select("EncounterEpicCsn", pl.col("Arrival_Instant").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f")),
	on="EncounterEpicCsn",
	how="left"
).with_columns(
	((pl.col("sepsis_1_dt")-pl.col("Arrival_Instant")).dt.total_hours(fractional=True)<=48).alias("POA")
).unique(subset=["EncounterEpicCsn", "sepsis_1_dt"])

/tmp/ipykernel_726851/1126798497.py:14: ChronoFormatWarning:

Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime



In [131]:
sepsis_1_filtered.sort(by=["EncounterEpicCsn", "sepsis_1_dt"])

EncounterEpicCsn,sepsis_1_dt,Arrival_Instant,POA
i64,datetime[μs],datetime[μs],bool
659308243,2022-06-14 22:10:00,2022-06-14 10:20:00,true
661424247,2022-09-01 10:45:00,2022-08-25 07:35:00,false
661471289,2022-06-07 19:01:00,2022-06-07 09:38:00,true
662152913,2022-08-15 14:10:00,2022-08-15 08:50:00,true
662358356,2022-06-13 03:16:00,2022-06-02 05:05:00,false
…,…,…,…
753558469,2026-05-24 18:39:00,2026-05-24 18:31:00,true
753563196,2026-05-25 04:49:00,2026-05-25 04:40:00,true
753573877,2026-05-25 20:45:00,2026-05-25 16:23:00,true


In [138]:
sepsis_1_raw.sort(by=["EncounterEpicCsn", "sepsis_1_dt"])

EncounterEpicCsn,sepsis_1_dt,Arrival_Instant,POA
i64,datetime[μs],datetime[μs],bool
659308243,2022-06-14 22:10:00,2022-06-14 10:20:00,true
661424247,2022-09-01 03:30:00,2022-08-25 07:35:00,false
661471289,2022-06-07 19:01:00,2022-06-07 09:38:00,true
662152913,2022-08-15 14:10:00,2022-08-15 08:50:00,true
662358356,2022-06-13 03:16:00,2022-06-02 05:05:00,false
…,…,…,…
753558469,2026-05-24 18:39:00,2026-05-24 18:31:00,true
753563196,2026-05-25 04:49:00,2026-05-25 04:40:00,true
753573877,2026-05-25 20:45:00,2026-05-25 16:23:00,true
